In [1]:
!pip install playwright
!playwright install chromium

!playwright install-deps
!playwright install chromium

!playwright install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 35.3 MB/s eta 0:00:00:00:0100:01
175.4 MiB [                    ] 0% 0.0s175.4 MiB [                    ] 0% 14.8s175.4 MiB [                    ] 0% 9.9s175.4 MiB [                    ] 0% 4.9s175.4 MiB [                    ] 1% 4.1s175.4 MiB [                    ] 2% 4.0s175.4 MiB [=                   ] 2% 3.7s175.4 MiB [=                   ] 3% 3.3s175.4 MiB [=                   ] 4% 2.9s175.4 MiB [=                   ] 5% 2.6s175.4 MiB [=                   ] 6% 2.6s175.4 MiB [=                   ] 6% 2.8s175.4 MiB [==                  ] 7% 2.6s175.4 MiB [==                  ] 8% 2.4s175.4 MiB [==                  ] 9% 2.3s175.4 MiB [==                  ] 10% 2.2s175.4 MiB [==                  ] 11% 2.1s175.4 MiB [===                 ] 12% 2.1s175.4 MiB [===                 ] 13% 2.0s175.4 MiB [===                 ] 14% 1.9s175.4 MiB [===                 ] 15% 1.9s175.4 MiB [===                 ] 16% 1.8s175.4 MiB [====       

In [2]:
import asyncio
from playwright.async_api import async_playwright

url = "https://www.behance.net/gallery/244803829/PORTFOLIO-2026-GRAPHIC-DESIGNER-ILLUSTRATOR"

async def run():
    async with async_playwright() as p:
        print("Launching browser...")
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        print("Navigating to page...")
        await page.goto(url, wait_until="domcontentloaded")
        
        # Give the initial page layout a moment to settle
        await asyncio.sleep(3)
        
        # --- Scroll down slowly to trigger lazy loading ---
        print("Scrolling down to load all images...")
        current_scroll_position = 0
        scroll_speed = 800  # pixels per jump
        
        while True:
            # Get the total height of the page dynamically
            total_height = await page.evaluate("document.body.scrollHeight")
            
            # Scroll down by a chunk
            current_scroll_position += scroll_speed
            await page.evaluate(f"window.scrollTo(0, {current_scroll_position});")
            
            # Wait briefly for assets to trigger loading
            await asyncio.sleep(0.5)
            
            # Break the loop once we hit the absolute bottom
            if current_scroll_position >= total_height:
                break
                
        print("Waiting a final moment for last images to complete...")
        await asyncio.sleep(5) 
        
        # Save the PDF
        print("Generating complete PDF...")
        await page.pdf(path="portfolio.pdf", format="A4", print_background=True)
        
        # Clean up
        await browser.close()
        print("Complete PDF Saved successfully with all images!")

# Execute in notebook
await run()

Launching browser...
Navigating to page...
Scrolling down to load all images...
Waiting a final moment for last images to complete...
Generating complete PDF...
Complete PDF Saved successfully with all images!


In [3]:
from IPython.display import IFrame

# Replace with your actual local file path or web URL
pdf_path = "portfolio.pdf" 

# Displays the PDF in a 1000x600 pixel window
IFrame(pdf_path, width=1000, height=600)

## Dividing the pdf into pages

In [4]:
pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 69.5 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
import fitz

pdf = fitz.open("portfolio.pdf")

chunk_size = 5

for start in range(0, pdf.page_count, chunk_size):

    end = min(start + chunk_size, pdf.page_count)

    new_pdf = fitz.open()

    for page_num in range(start, end):
        new_pdf.insert_pdf(
            pdf,
            from_page=page_num,
            to_page=page_num
        )

    new_pdf.save(
        f"chunk_{start+1}_{end}.pdf"
    )

## Converting pdf to image

In [6]:
pip install pdf2image

Note: you may need to restart the kernel to use updated packages.


In [7]:
import fitz
import os

pdf = fitz.open("chunk_1_5.pdf")

os.makedirs("images", exist_ok=True)

for i in range(len(pdf)):

    page = pdf[i]

    pix = page.get_pixmap(
        matrix=fitz.Matrix(2, 2)
    )

    pix.save(
        f"images/page_{i+1}.png"
    )

## VLM

In [8]:
pip install transformers accelerate qwen-vl-utils pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 55.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda

In [9]:
!pip install -U bitsandbytes>=0.46.1

In [10]:
from transformers import Qwen2_5_VLForConditionalGeneration
from transformers import AutoProcessor

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

from transformers import BitsAndBytesConfig
import torch

# 1. This tells Kaggle to compress the model so it takes up 75% less memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 2. Load the model using that compression setting
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
from PIL import Image

image = Image.open("images/page_1.png")

In [12]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image,
            },
            {
                "type": "text",
                "text": """
Analyze these portfolio pages.

Return JSON:

{
 "skills": [],
 "tools": [],
 "domains": [],
 "projects": []
}
"""
            }
        ]
    }
]

In [13]:
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=[text],
    images=[image],
    padding=True,
    return_tensors="pt"
)

inputs = inputs.to(model.device)

In [14]:
generated_ids = model.generate(
    **inputs,
    max_new_tokens=500,
    use_cache=True
)

output = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

print(output[0])

system
You are a helpful assistant.
user

Analyze these portfolio pages.

Return JSON:

{
 "skills": [],
 "tools": [],
 "domains": [],
 "projects": []
}

assistant
```json
{
  "skills": ["Graphic Design", "Illustration"],
  "tools": ["Adobe Photoshop", "Adobe Illustrator"],
  "domains": ["Branding", "Web Design", "Logo Design"],
  "projects": [
    {
      "title": "Unboxing My Portfolio",
      "description": "A shipping box with various stickers and text elements, showcasing the designer's skills in graphic design and illustration."
    },
    {
      "title": "Branding Project",
      "description": "A project that involved creating a brand identity for a fictional company, including logos, business cards, and promotional materials."
    },
    {
      "title": "Web Design Project",
      "description": "A project that involved designing a website for a fictional e-commerce store, including navigation, layout, and user interface elements."
    }
  ]
}
```
